# The Murder Game
---

**Investigares:**

* Danillo Barros de Souza
* Mikel Añibarro
* Naia merino
* Juan Carlos Piña
---

**Objetivo:** 

* Para este ejercicio vas a ponerte la gabardina y el sombrero ya que tendrás que investigar un
asesinato en SQL City.
* Aplicarás los conceptos básicos de SQL y aprenderás a manejarte con un modelo de datos,
mientras resuelves un caso de asesinato :)
Este espectacular juego está creado por Joon Park y Cathy He, y podrás encontrar todos los
detalles en su web
* Para realizar el ejercicio tienes dos opciones, o bien desde la propia web, donde encontrarás un
intérprete de sentencias SQL interactivo o lo que es más recomendable, atacar a la base de
datos que tienes el directorio de data, ya sea desde Python o Javascript o ya sea desde
SQLiteStudio u otra similar

---
**Resumen:**

1. Hay dos testigos importantes con sus testimonios: **Annabel Miller** y **Morty Schapiro**;
2. Annabel vio el crimen y reconoció al asesino en el gimnasio donde trabaja. Sospechosos: **Jeremy Bowers** y **Joe Germuskua**
3. Morty vio al sospechoso salir del encenado del crimen en un coche y tiene detalles de la matrícula; Sospechosos: **Jeremy Bowers** y **Tushar Chandra**.
4. El único sospechoso que cumple con las informaciones de 2. y 3. es **Jeremy Bowers**;
5. En su testimonio, Jeremy comenta que fue contratado (no exactamente para cometer un crimen), según él, por una mujer con mucha pasta y la describe ( modelo de coche, y que ha asistido a un evento de SQL City tres veces en diciembre de 2017);
6. Comprobamos quién estuvo 3 veces en el evento de SQL city y tiene un Tesla modelo S.
7. La mujer en la descripción de Jeremy se trata de **Miranda Priestly**.
8. Jeremy estuvo en un evento de Facebook en el día del crimen (¿lo cometió ahí? ¿Dónde pasó el evento?)
10. Se puede confirmar que de hecho Miranda tiene mucha pasta.
11. Miranda **no** ha dado su testimonio.
---
# Concusión:
* La autora intelectual sería:
`Miranda Priestly`.
* El probable autor material sería:
`Jeremy Bowers`.

---
Abajo, las informaciones de nuestra búsqueda por el asesino:

```sql
-- ver la pinta que tiene
SELECT * 
FROM crime_scene_report;
SELECT description 
FROM crime_scene_report
WHERE date = '20180115' AND type = 'murder' and city = 'SQL City';
SELECT * FROM PERSON;

-- SACAMOS LA INFO DEL TESTIGO 1
SELECT *
FROM person
WHERE address_street_name LIKE 'Northwestern%'
ORDER BY address_number DESC
LIMIT 1;

-- SACAMOS LA INFO DEL TESTIGO 2
SELECT *
FROM person
WHERE name LIKE 'Annabel%'
  AND address_street_name LIKE 'Franklin%';
  
--vamos a ver qué han dicho
SELECT person_id, transcript FROM interview
WHERE person_ID IN ('14887','16371');

--buscamos la información que corresponde a lo que indica un testigo. 
SELECT name, person_id FROM get_fit_now_member AS GM
INNER JOIN get_fit_now_check_in AS GC ON GM.id = GC.membership_id
WHERE GC.membership_id LIKE '48Z%' and GM.membership_status = 'gold';

--salen dos. Compmrobamos la matricula
SELECT *  FROM person
INNER JOIN drivers_license ON person.license_id = drivers_license.id
WHERE drivers_license.plate_number LIKE '%H42W%' AND gender = 'male';

-- sale que Jeremy Bowers tiene ese coche pero también otra persona Tushar Chandra
--vamos con el otro testimonio
SELECT name, person_id, GC.check_in_time, GC.check_out_time FROM get_fit_now_member AS GM
INNER JOIN get_fit_now_check_in AS GC ON GM.id = GC.membership_id
WHERE GM.membership_status ='gold' AND GC.membership_id LIKE '48Z%' AND   GC.check_in_date = '20180109';
--Joe Germuska
--Jeremy Bowers
--añadimos las horas para ver cuanto tiempo están
--vamos a ver las entrevistas a los sospechosos

SELECT person.name,person_id, transcript FROM interview
INNER JOIN person ON person.id = interview.person_id
WHERE person.name IN ('Joe Germuska','Jeremy Bowers','Tushar Chandra');
--Parece que le ha contratado una mujer como sicario. Vamos a ver quien es esa mujer

-- empezamos por el coche y el evento
SELECT person.name FROM person
INNER JOIN drivers_license ON person.license_id = drivers_license.id
INNER JOIN facebook_event_checkin ON person.id = facebook_event_checkin.person_id
WHERE drivers_license.car_make = 'Tesla' AND drivers_license.car_model = 'Model S' AND facebook_event_checkin.event_name = 'SQL Symphony Concert';

--solo sale Miranda Priestly. Parece la autora intelectual. Vamos a comprobar
SELECT annual_income, person.id FROM income
INNER JOIN person ON person.ssn = income.ssn
WHERE person.name = 'Miranda Priestly'
    ;
--el equipo investigador tiene dudas de si 310000 pueden considerarse altos ingresos. 
-- Vamos a ver la media de los ingresos
-- 
SELECT AVG(annual_income) FROM income;

--Parece nuestra chica. A ver qué dice
SELECT transcript FROM interview
WHERE person_id = '99716';

--no ha dado su testimonio
-- vamos a investigar a Jerermy
SELECT * FROM facebook_event_checkin
INNER JOIN person ON person.id = facebook_event_checkin.person_id
WHERE person.name = 'Jeremy Bowers';

--En la fecha del crimen el autor material está en un evento. No sabemos donde es este evento. ¿Será en SQL city?
-- Con la información que tenemos es autor material es Jeremy Bowers
-- Y la autora intelectual Miranda Priestly

INSERT INTO solution VALUES (1, 'Jeremy Bowers');
        SELECT value FROM solution;

```

# Importando librerías necesárias:

In [2]:
import pandas as pd
import sqlite3

# Enunciado Original: 

A crime has taken place and the detective needs your help. The detective gave you the crime
scene report, but you somehow lost it. You vaguely remember that the crime was a ***murder*** that
occurred sometime on ***Jan.15, 2018*** and that it took place in ***SQL City***. Start by retrieving the
corresponding crime scene report from the police department’s database


![Scheme](Scheme_murder_game.png)

# Conectandose a la basis de datos:

In [3]:
connection = sqlite3.connect("sql-murder-mystery.db")

# Leyendo dataset:

In [4]:
G = pd.read_sql_query("SELECT * FROM get_fit_now_check_in", connection)

In [5]:
G.head()

,membership_id,check_in_date,check_in_time,check_out_time
0,NL318,20180212,329,365
1,NL318,20170811,469,920
2,NL318,20180429,506,554
3,NL318,20180128,124,759
4,NL318,20171027,418,1019


In [6]:
F = pd.read_sql_query("SELECT * FROM facebook_event_checkin", connection)
F["date"]=pd.to_datetime(F["date"], format='%Y%m%d')

I = pd.read_sql_query("SELECT * FROM interview ", connection)
C = pd.read_sql_query("SELECT * FROM crime_scene_report", connection)
D=pd.read_sql_query("SELECT * FROM drivers_license",connection)
G = pd.read_sql_query("SELECT * FROM get_fit_now_check_in", connection)
G["check_in_date"] = pd.to_datetime(G["check_in_date"], format='%Y%m%d')
G1 = pd.read_sql_query("SELECT * FROM get_fit_now_member", connection)
P = pd.read_sql_query("SELECT * FROM person", connection)
In=pd.read_sql_query("SELECT * FROM income",connection)

In [8]:
D.car_model.unique()

array(['MDX', 'SRX', 'xB', 'Rogue', 'GS', 'Sportage', '9-5', 'XK',
       'Sierra Denali', 'Sierra 2500', 'S60', 'Intrigue', 'Express 1500',
       'Matrix', 'Azera', 'Pathfinder', '350Z', 'E-Series', 'Savana 3500',
       'S-Series', 'Ram Van 3500', 'F350', 'Grand Vitara', 'LaCrosse',
       'Regal', 'Trailblazer', 'Savana 1500', 'Insight', 'Cutlass',
       'Focus', 'Ram Van 2500', '9-7X', 'VehiCROSS', 'E250',
       'F-Series Super Duty', 'Corolla', 'Freelander', 'S-Class',
       'Charger', 'Ram 3500', 'Envoy', 'XC90', 'Discovery', 'TSX',
       'SL-Class', 'Range Rover Sport', 'Montero', 'Silhouette',
       'Mazdaspeed 3', '760', 'Grand Cherokee', 'Suburban 1500', 'V70',
       'C-Class', 'Liberty', 'Armada', 'Navigator', 'Murano', 'Cabriolet',
       'Discovery Series II', 'Monte Carlo', 'CX-9', 'M-Class', 'Escape',
       'Dakota', 'Pajero', 'Rabbit', 'XL-7', 'Cavalier', 'F150',
       'Forester', 'Aura', 'R-Class', 'Thunderbird', 'Cherokee', 'G',
       'Silverado 2500', 'Taho

In [4]:
C.head(5)

,date,type,description,city
0,20180115,robbery,A Man Dressed as Spider-Man Is on a Robbery Spree,NYC
1,20180115,murder,Life? Dont talk to me about life.,Albany
2,20180115,murder,"Mama, I killed a man, put a gun against his he...",Reno
3,20180215,murder,REDACTED REDACTED REDACTED,SQL City
4,20180215,murder,Someone killed the guard! He took an arrow to ...,SQL City


# Informaciones del crimen:
* Crimenes del tipo asesinato en SQL City:

In [11]:
C[(C.city=="SQL City")&(C["type"]=="murder")]["description"].values.tolist()

['REDACTED REDACTED REDACTED',
 'Someone killed the guard! He took an arrow to the knee!',
 'Security footage shows that there were 2 witnesses. The first witness lives at the last house on "Northwestern Dr". The second witness, named Annabel, lives somewhere on "Franklin Ave".']

In [12]:
P[P.name.str.contains("Annabel")]

,id,name,license_id,address_number,address_street_name,ssn
665,16371,Annabel Miller,490173,103,Franklin Ave,318771143
7601,78354,Annabell Siona,158932,978,Whitewater Dr,800278294
7650,78799,Annabell Droneburg,984316,1944,W Natalie Dr,478793500
8509,86541,Annabell Zwilling,709133,1859,Patti Rd,332961158


# Testigo 1: Annabel Miller

In [13]:
P[(P.name.str.contains("Annabel"))&(P.address_street_name=="Franklin Ave")]

,id,name,license_id,address_number,address_street_name,ssn
665,16371,Annabel Miller,490173,103,Franklin Ave,318771143


# Testigo 2: Morty Schapiro

In [14]:
P[(P.address_street_name=="Northwestern Dr")&(P.address_number==P.address_number.max())]

,id,name,license_id,address_number,address_street_name,ssn
499,14887,Morty Schapiro,118009,4919,Northwestern Dr,111564949


# Testimónio de los testigos:

### Testigo 1:

In [5]:
I[I.person_id==16371].transcript.values

array(['I saw the murder happen, and I recognized the killer from my gym when I was working out last week on January the 9th.'],
      dtype=object)

### Testigo 2:

In [6]:
I[I.person_id==14887].transcript.values

array(['I heard a gunshot and then saw a man run out. He had a "Get Fit Now Gym" bag. The membership number on the bag started with "48Z". Only gold members have those bags. The man got into a car with a plate that included "H42W".'],
      dtype=object)

# Cruzando datos del Gymnasion:

In [7]:
GG1=G1.rename(columns={"id":"membership_id"}).merge(G,on="membership_id")

# Cumplen con las descripciones de la víctima 1:
1. Membros Gold del gym con membership ID conteniendo "48Z";
2. Estaban este día en el gym el día 9 de enero de 2018;

# Suspechosos:
* Jeremy Bowers
* Joe Germuska

# Jusiticativa:

In [8]:
GG1[(GG1.membership_status=="gold")&GG1.membership_id.str.contains("48Z")&(GG1['check_in_date'].dt.year == 2018)&(GG1['check_in_date'].dt.month == 1)&(GG1['check_in_date'].dt.day == 9)]

,membership_id,person_id,name,membership_start_date,membership_status,check_in_date,check_in_time,check_out_time
2700,48Z7A,28819,Joe Germuska,20160305,gold,2018-01-09,1600,1730
2701,48Z55,67318,Jeremy Bowers,20160101,gold,2018-01-09,1530,1700


# Cruzando datos de identificación:

In [13]:
PD=D.rename(columns={"id":"license_id"}).merge(P,on="license_id").rename(columns={"id":"person_id"})

In [14]:
PD.head(5)

,license_id,age,height,eye_color,hair_color,gender,plate_number,car_make,car_model,person_id,name,address_number,address_street_name,ssn
0,100280,72,57,brown,red,male,P24L4U,Acura,MDX,22757,Alfonzo Lighter,2380,Demeo Rd,158688705
1,100460,63,72,brown,brown,female,XF02T6,Cadillac,SRX,84910,Jayme Secor,872,Cooksey Circle,221254409
2,101029,62,74,green,green,female,VKY5KR,Scion,xB,84921,Cassey Boeve,3698,Pottawattami Blvd,377934170
3,101198,43,54,amber,brown,female,Y5NZ08,Nissan,Rogue,58014,Logan Helde,929,Gilton St,545228106
4,101255,18,79,blue,grey,female,5162Z1,Lexus,GS,90704,Shanna Zingone,2071,N Village Rd,215917206


# Cumplen con la descripción de la víctima 2:

1. El número de matrícula del coche contiene "H42W";
2. Son Varones.

# Suspechosos:

* Jeremy Bowers;
* Tushar Chandra

# Justificativa:

In [31]:
PD[(PD.plate_number.str.contains("H42W"))&(PD.gender=="male")]

,license_id,age,height,eye_color,hair_color,gender,plate_number,car_make,car_model,person_id,name,address_number,address_street_name,ssn
3528,423327,30,70,brown,brown,male,0H42W2,Chevrolet,Spark LS,67318,Jeremy Bowers,530,"Washington Pl, Apt 3A",871539279
6239,664760,21,71,black,black,male,4H42WR,Nissan,Altima,51739,Tushar Chandra,312,Phi St,137882671


In [15]:
data= PD.merge(GG1,on=["person_id","name"])

# CONCLUSIÓN: 
* # Jeremy Bowers cumple con la descripción de los dos testigos:

In [16]:
I

,person_id,transcript
0,28508,‘I deny it!’ said the March Hare.\n
1,63713,\n
2,86208,"way, and the whole party swam to the shore.\n"
3,35267,"lessons in here? Why, there’s hardly room for ..."
4,33856,\n
...,...,...
4986,37357,Alice did not wish to offend the Dormouse agai...
4987,10206,"time,’ she said, ‘than waste it in asking ridd..."
4988,14887,I heard a gunshot and then saw a man run out. ...
4989,16371,"I saw the murder happen, and I recognized the ..."


In [17]:
Fdata=F.merge(data[["name","person_id"]],on="person_id")

In [18]:
Fdata[Fdata.name=="Jeremy Bowers"]



,person_id,event_id,event_name,date,name
5272,67318,4719,The Funky Grooves Tour,2018-01-15,Jeremy Bowers
5273,67318,1143,SQL Symphony Concert,2017-12-06,Jeremy Bowers


In [132]:
F

,person_id,event_id,event_name,date
0,28508,5880,Nudists are people who wear one-button suits.\n,2017-09-13
1,63713,3865,but that's because it's the best book on anyth...,2017-10-09
2,63713,3999,"If Murphy's Law can go wrong, it will.\n",2017-05-02
3,63713,6436,Old programmers never die. They just branch t...,2017-09-26
4,82998,4470,Help a swallow land at Capistrano.\n,2017-10-22
...,...,...,...,...
20006,99716,1143,SQL Symphony Concert,2017-12-06
20007,99716,1143,SQL Symphony Concert,2017-12-12
20008,99716,1143,SQL Symphony Concert,2017-12-29
20009,67318,4719,The Funky Grooves Tour,2018-01-15


# CURIOSIDAD:
* # Jeremy Bowers estuvo en el evento el 15 de enero:

# CONCLUSÍON: Jeremy todavía és suspechoso, pero

* # No se sabe si el evento que estaba en esta fecha se pasó en SQL city.

In [47]:
data

,license_id,age,height,eye_color,hair_color,gender,plate_number,car_make,car_model,person_id,name,address_number,address_street_name,ssn,membership_id,membership_start_date,membership_status,check_in_date,check_in_time,check_out_time
0,108374,18,63,brown,red,male,X2KE6N,Ford,Escape,45746,Frederic Borsellino,3683,Stattel Dr,355865979,5Y85X,20170803,regular,2017-12-11,67,460
1,108374,18,63,brown,red,male,X2KE6N,Ford,Escape,45746,Frederic Borsellino,3683,Stattel Dr,355865979,5Y85X,20170803,regular,2017-09-13,1071,1146
2,108374,18,63,brown,red,male,X2KE6N,Ford,Escape,45746,Frederic Borsellino,3683,Stattel Dr,355865979,5Y85X,20170803,regular,2017-05-16,1033,1141
3,108374,18,63,brown,red,male,X2KE6N,Ford,Escape,45746,Frederic Borsellino,3683,Stattel Dr,355865979,5Y85X,20170803,regular,2017-08-14,10,335
4,108374,18,63,brown,red,male,X2KE6N,Ford,Escape,45746,Frederic Borsellino,3683,Stattel Dr,355865979,5Y85X,20170803,regular,2017-05-23,390,643
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2697,990930,19,55,blue,brown,female,78B71X,Daewoo,Leganza,46526,Carolin Orellano,3341,Ferry Point St,352639412,I733Y,20180325,silver,2017-10-28,662,766
2698,990930,19,55,blue,brown,female,78B71X,Daewoo,Leganza,46526,Carolin Orellano,3341,Ferry Point St,352639412,I733Y,20180325,silver,2017-11-12,609,837
2699,990930,19,55,blue,brown,female,78B71X,Daewoo,Leganza,46526,Carolin Orellano,3341,Ferry Point St,352639412,I733Y,20180325,silver,2017-01-10,1130,1178
2700,990930,19,55,blue,brown,female,78B71X,Daewoo,Leganza,46526,Carolin Orellano,3341,Ferry Point St,352639412,I733Y,20180325,silver,2018-03-01,54,786


In [49]:
Idata=I.merge(data.rename(columns={"id":"person_id"}),on="person_id")

In [103]:
Fdata=F.merge(data[["name","person_id"]],on="person_id")

In [104]:
data[(data.plate_number.str.contains("H42W"))&(data.gender=="male")]

,license_id,age,height,eye_color,hair_color,gender,plate_number,car_make,car_model,person_id,name,address_number,address_street_name,ssn,membership_id,membership_start_date,membership_status,check_in_date,check_in_time,check_out_time
771,423327,30,70,brown,brown,male,0H42W2,Chevrolet,Spark LS,67318,Jeremy Bowers,530,"Washington Pl, Apt 3A",871539279,48Z55,20160101,gold,2018-01-09,1530,1700


# Testigo de Jeremy:

In [53]:
Idata[Idata.name.str.contains("Jeremy")].transcript.values

array(['I was hired by a woman with a lot of money. I don\'t know her name but I know she\'s around 5\'5" (65") or 5\'7" (67"). She has red hair and she drives a Tesla Model S. I know that she attended the SQL Symphony Concert 3 times in December 2017.\n'],
      dtype=object)

In [69]:
D1=D[(D["car_make"].str.contains("Tesla"))&(D.car_model=="Model S")&(D.gender=="female")].rename(columns={"id":"license_id"})

In [89]:
P1=P.merge(D1,on="license_id").rename(columns={"id":"person_id"})

In [90]:
P1

,person_id,name,license_id,address_number,address_street_name,ssn,age,height,eye_color,hair_color,gender,plate_number,car_make,car_model
0,67673,Alyssa Faggett,682231,3838,Mc Guffie Dr,666309770,51,80,green,grey,female,3Z0M62,Tesla,Model S
1,78881,Red Korb,918773,107,Camerata Dr,961388910,48,65,black,red,female,917UU3,Tesla,Model S
2,90700,Regina George,291182,332,Maple Ave,337169072,65,66,blue,red,female,08CM64,Tesla,Model S
3,99716,Miranda Priestly,202298,1883,Golden Ave,987756388,68,66,green,red,female,500123,Tesla,Model S


In [92]:
F[(F.person_id.isin(P1.person_id))&(F.event_name.str.contains("SQL"))]

,person_id,event_id,event_name,date
20006,99716,1143,SQL Symphony Concert,2017-12-06
20007,99716,1143,SQL Symphony Concert,2017-12-12
20008,99716,1143,SQL Symphony Concert,2017-12-29


# Conclusión: Jeremy hablava de Miranda Priestly

In [102]:
P[P.name.str.contains("Miranda")]

,id,name,license_id,address_number,address_street_name,ssn
9985,99716,Miranda Priestly,202298,1883,Golden Ave,987756388


In [36]:
data[(data.plate_number.str.contains("H42W"))&(data.gender=="male")]

,license_id,age,height,eye_color,hair_color,gender,plate_number,car_make,car_model,person_id,name,address_number,address_street_name,ssn,membership_id,membership_start_date,membership_status,check_in_date,check_in_time,check_out_time
771,423327,30,70,brown,brown,male,0H42W2,Chevrolet,Spark LS,67318,Jeremy Bowers,530,"Washington Pl, Apt 3A",871539279,48Z55,20160101,gold,2018-01-09,1530,1700


In [108]:
P[P.name.str.contains("Miranda")]

,id,name,license_id,address_number,address_street_name,ssn
9985,99716,Miranda Priestly,202298,1883,Golden Ave,987756388


In [113]:
I[I.person_id==99716]

,person_id,transcript


# Conclusión:
* # Miranda no ha dado su testimónio
* # Miranda de hecho tiene mucha pasta, como había dicho Jeremy.

In [116]:
In[In.ssn==987756388]

,ssn,annual_income
7422,987756388,310000


In [118]:
P[P.name.str.contains("Jeremy B")]

,id,name,license_id,address_number,address_street_name,ssn
6327,67318,Jeremy Bowers,423327,530,"Washington Pl, Apt 3A",871539279


In [119]:
In[In.ssn==871539279]

,ssn,annual_income
6474,871539279,10500


In [120]:
P[P.name.str.contains("Annabel")]

,id,name,license_id,address_number,address_street_name,ssn
665,16371,Annabel Miller,490173,103,Franklin Ave,318771143
7601,78354,Annabell Siona,158932,978,Whitewater Dr,800278294
7650,78799,Annabell Droneburg,984316,1944,W Natalie Dr,478793500
8509,86541,Annabell Zwilling,709133,1859,Patti Rd,332961158


In [121]:
In[In.ssn==318771143]

,ssn,annual_income


In [123]:
P[P.name.str.contains("Tushar Chandra")]


,id,name,license_id,address_number,address_street_name,ssn
4664,51739,Tushar Chandra,664760,312,Phi St,137882671


In [124]:
In[In.ssn==137882671]

,ssn,annual_income


In [125]:
P[P.name.str.contains("Joe Germuska")]


,id,name,license_id,address_number,address_street_name,ssn
2037,28819,Joe Germuska,173289,111,Fisk Rd,138909730


In [126]:
In[In.ssn==138909730]

,ssn,annual_income


In [137]:
F[F.person_id==67318]

,person_id,event_id,event_name,date
20009,67318,4719,The Funky Grooves Tour,2018-01-15
20010,67318,1143,SQL Symphony Concert,2017-12-06
